# NYC Transit Recovery - Data Exploration

**CSE 6242 - Data and Visual Analytics**

This notebook explores the datasets and performs initial analysis for the NYC transit recovery project.

In [ ]:
# Standard imports
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Plotly for interactive visualizations
import plotly.express as px
import plotly.graph_objects as go

# Settings
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

# Project paths
PROJECT_ROOT = Path('.').parent
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_EXTERNAL = PROJECT_ROOT / 'data' / 'external'

print(f"Project root: {PROJECT_ROOT.absolute()}")

## 1. Load NTA Boundaries

In [ ]:
# Load NTA boundaries
nta_gdf = gpd.read_file(DATA_EXTERNAL / 'nta_boundaries_2020.geojson')
print(f"Loaded {len(nta_gdf)} NTAs")
nta_gdf.head()

In [ ]:
# Plot NTA boundaries
fig, ax = plt.subplots(figsize=(12, 14))
nta_gdf.plot(ax=ax, edgecolor='black', linewidth=0.5, facecolor='lightblue', alpha=0.5)
ax.set_title('NYC Neighborhood Tabulation Areas (NTAs)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Load MTA Ridership Data

In [ ]:
# Load MTA ridership (this may take a minute due to file size)
mta_path = DATA_RAW / 'mta_subway_hourly_ridership_2020_2024.csv'

if mta_path.exists():
    print("Loading MTA ridership data...")
    mta_df = pd.read_csv(mta_path, parse_dates=['transit_timestamp'], nrows=1000000)  # Sample for exploration
    print(f"Loaded {len(mta_df):,} rows (sample)")
    display(mta_df.head())
else:
    print(f"MTA data not found at {mta_path}")
    print("Run: python src/data/download_all.py")

In [ ]:
# Basic statistics
if 'mta_df' in dir():
    print("\nColumn info:")
    print(mta_df.dtypes)
    
    print("\nDate range:")
    print(f"  Min: {mta_df['transit_timestamp'].min()}")
    print(f"  Max: {mta_df['transit_timestamp'].max()}")
    
    print(f"\nUnique stations: {mta_df['station_complex'].nunique()}")

## 3. Aggregate Ridership Over Time

In [ ]:
if 'mta_df' in dir():
    # Aggregate to daily
    mta_df['date'] = mta_df['transit_timestamp'].dt.date
    daily_ridership = mta_df.groupby('date')['ridership'].sum().reset_index()
    daily_ridership['date'] = pd.to_datetime(daily_ridership['date'])
    
    # Plot
    fig = px.line(daily_ridership, x='date', y='ridership',
                  title='NYC Subway Daily Ridership (Sample)')
    fig.update_layout(xaxis_title='Date', yaxis_title='Total Ridership')
    fig.show()

## 4. Station Location Analysis

In [ ]:
if 'mta_df' in dir():
    # Get unique stations with coordinates
    stations = mta_df[['station_complex_id', 'station_complex', 'latitude', 'longitude']].drop_duplicates()
    stations = stations.dropna(subset=['latitude', 'longitude'])
    
    print(f"Unique stations with coordinates: {len(stations)}")
    
    # Create station GeoDataFrame
    stations_gdf = gpd.GeoDataFrame(
        stations,
        geometry=gpd.points_from_xy(stations['longitude'], stations['latitude']),
        crs='EPSG:4326'
    )
    
    # Plot stations on NTA map
    fig, ax = plt.subplots(figsize=(12, 14))
    nta_gdf.plot(ax=ax, edgecolor='gray', linewidth=0.3, facecolor='lightgray', alpha=0.5)
    stations_gdf.plot(ax=ax, color='red', markersize=5, alpha=0.7)
    ax.set_title('MTA Subway Stations and NTA Boundaries', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Spatial Join: Stations to NTAs

In [ ]:
if 'stations_gdf' in dir():
    # Ensure same CRS
    nta_gdf = nta_gdf.to_crs('EPSG:4326')
    
    # Spatial join (use 2020 NTA column names: nta2020, ntaname)
    stations_with_nta = gpd.sjoin(stations_gdf, nta_gdf[['nta2020', 'ntaname', 'geometry']], 
                                   how='left', predicate='within')
    
    print(f"Stations matched to NTAs: {stations_with_nta['nta2020'].notna().sum()}")
    print(f"Stations not matched: {stations_with_nta['nta2020'].isna().sum()}")
    
    # Stations per NTA
    stations_per_nta = stations_with_nta.groupby('nta2020').size().sort_values(ascending=False)
    print(f"\nTop 10 NTAs by station count:")
    print(stations_per_nta.head(10))

## 6. Load Processed Data (if available)

In [ ]:
# Try to load processed analysis data
processed_path = DATA_PROCESSED / 'nta_analysis_ready.geojson'

if processed_path.exists():
    print("Loading processed analysis data...")
    analysis_gdf = gpd.read_file(processed_path)
    print(f"Loaded {len(analysis_gdf)} NTAs with {len(analysis_gdf.columns)} features")
    display(analysis_gdf.head())
else:
    print(f"Processed data not found at {processed_path}")
    print("Run: python src/data/process_data.py")

In [ ]:
# Plot recovery index if available
if 'analysis_gdf' in dir() and 'recovery_index' in analysis_gdf.columns:
    fig, ax = plt.subplots(figsize=(12, 14))
    analysis_gdf.plot(column='recovery_index', cmap='RdYlGn', legend=True,
                      legend_kwds={'label': 'Recovery Index'}, ax=ax,
                      edgecolor='white', linewidth=0.3)
    ax.set_title('NYC Subway Ridership Recovery Index by Neighborhood', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"\nRecovery Index Statistics:")
    print(analysis_gdf['recovery_index'].describe())

## 7. Next Steps

1. **Download full data**: Run `python src/data/download_all.py`
2. **Process data**: Run `python src/data/process_data.py`
3. **Run analysis**: Run `python src/analysis/recovery_analysis.py`
4. **Launch dashboard**: Run `python src/visualization/dashboard.py`